# FEDS Benchmark: FedAvg vs DSFL vs FEDS

Complete benchmark comparing three federated learning methods:
1. **FedAvg** - Standard federated averaging (baseline)
2. **DSFL** - Dynamic Sparsified FL with fixed K
3. **FEDS** - Federated learning with adaptive K (loss-feedback)

**Supported Datasets:** MNIST, CIFAR-10, Speech Commands

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
DATASET = "MNIST"  # Options: "MNIST", "CIFAR10", "Speech"

NUM_CLIENTS = 10
NUM_ROUNDS = 300  # Reduce for quick tests (e.g., 10-20)

# Which methods to run
RUN_FEDAVG = True   # Baseline
RUN_DSFL = True     # Fixed K sparsification
RUN_FEDS = True     # Adaptive K (our method)
# ============================================================
print(f"Benchmark on: {DATASET}")
print(f"Methods: FedAvg={RUN_FEDAVG}, DSFL={RUN_DSFL}, FEDS={RUN_FEDS}")

In [ ]:
# Setup: Clone repo and install dependencies
print("[SETUP] Installing dependencies...")
import os, sys, subprocess, time, json, signal, threading
import numpy as np
import matplotlib.pyplot as plt

# Clone repository
if not os.path.exists("feds"):
    print("[SETUP] Cloning repository...")
    !git clone https://github.com/masud1901/feds.git
else:
    print("[SETUP] Repository already exists")

os.chdir("feds")
sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.join(os.getcwd(), "utils"))

# Install dependencies
!pip install -q flwr torch torchvision torchaudio numpy tensorboard matplotlib scipy

import flwr
import torch
print(f"[SETUP] flwr: {flwr.__version__}")
print(f"[SETUP] torch: {torch.__version__}")
print(f"[SETUP] GPU: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"[SETUP] Working directory: {os.getcwd()}")
print("[SETUP] Setup complete!")

In [ ]:
# Create initial model artifacts
print("[ARTIFACTS] Creating model artifacts...")
artifact_scripts = {
    "MNIST": "scripts/create_artifacts_colab.py",
    "CIFAR10": "scripts/create_artifacts_CIFAR.py",
    "Speech": "scripts/create_artifacts_Speech.py"
}

script = artifact_scripts[DATASET]
print(f"[ARTIFACTS] Running: {script}")
!python {script}

# Environment setup
env = os.environ.copy()
env["PYTHONPATH"] = f"{os.getcwd()}:{os.path.join(os.getcwd(), 'utils')}"
env["FEDS_DATASET"] = DATASET
print("[ARTIFACTS] Done!")

In [ ]:
# Helper function to stream output
def stream_output(proc, prefix=""):
    """Stream subprocess output in real-time."""
    for line in iter(proc.stdout.readline, ""):
        print(f"{prefix}{line}", end="", flush=True)

def run_experiment(server_script, method_name):
    """Run a single experiment with full output."""
    print(f"\n{'='*70}")
    print(f"  STARTING: {method_name} on {DATASET}")
    print(f"{'='*70}\n")
    
    # Clean up old files
    for f in ['feds_k_tracker.json', 'feds_history.json', 'feds_k_trajectory.json', 'dsfl_history.json']:
        if os.path.exists(f):
            os.remove(f)
            print(f"[CLEANUP] Removed {f}")
    
    print(f"[SERVER] Starting {server_script}...")
    
    # Start server with output visible
    server = subprocess.Popen(
        [sys.executable, server_script],
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    
    # Stream server output in background thread
    server_thread = threading.Thread(target=stream_output, args=(server, "[SERVER] "))
    server_thread.daemon = True
    server_thread.start()
    
    print(f"[SERVER] Waiting for server to initialize...")
    time.sleep(8)
    print(f"[SERVER] Server should be ready!\n")
    
    # Start clients
    client_scripts = {
        "MNIST": "clients/client-MNIST.py",
        "CIFAR10": "clients/client-CIFAR.py",
        "Speech": "clients/client-SpeechCommands.py"
    }
    client_script = client_scripts[DATASET]
    
    print(f"[CLIENTS] Starting {NUM_CLIENTS} clients...")
    procs = []
    for i in range(NUM_CLIENTS):
        p = subprocess.Popen(
            [sys.executable, client_script, "--seed", str(i)],
            env=env,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )
        procs.append(p)
        print(f"[CLIENTS] Started client {i} (PID: {p.pid})")
    
    print(f"\n[CLIENTS] All clients started. Waiting for training to complete...")
    print(f"[CLIENTS] (Server output will show round progress below)\n")
    
    # Wait for all clients to finish
    finished = 0
    for i, p in enumerate(procs):
        p.wait()
        finished += 1
        print(f"[CLIENTS] Client {i} finished ({finished}/{NUM_CLIENTS})")
    
    print(f"\n[CLIENTS] All clients finished!")
    
    # Stop server
    print(f"[SERVER] Stopping server...")
    time.sleep(2)
    server.terminate()
    server.wait()
    
    print(f"\n{'='*70}")
    print(f"  COMPLETED: {method_name}")
    print(f"{'='*70}\n")
    return True

In [ ]:
# Run FedAvg Baseline
if RUN_FEDAVG:
    run_experiment("server_FedAvg.py", "FedAvg (Baseline)")

In [ ]:
# Run DSFL (Fixed K)
if RUN_DSFL:
    run_experiment("server_DSFL.py", "DSFL (Fixed K)")

In [ ]:
# Run FEDS (Adaptive K)
if RUN_FEDS:
    run_experiment("server.py", "FEDS (Adaptive K)")

In [ ]:
# Load results and generate comparison plots
print("[RESULTS] Loading experiment results...")

# Load results for each method
results = {}

fedavg_file = f"results_FedAvg_{DATASET}.json"
dsfl_file = f"results_DSFL_{DATASET}.json"

if os.path.exists(fedavg_file):
    with open(fedavg_file) as f:
        results['FedAvg'] = json.load(f)
    print(f"[RESULTS] Loaded FedAvg: {len(results['FedAvg'].get('rounds', []))} rounds")

if os.path.exists(dsfl_file):
    with open(dsfl_file) as f:
        results['DSFL'] = json.load(f)
    print(f"[RESULTS] Loaded DSFL: {len(results['DSFL'].get('rounds', []))} rounds")

if os.path.exists('feds_k_tracker.json'):
    with open('feds_k_tracker.json') as f:
        k_data = json.load(f)
    # Create FEDS results from k_tracker loss_history
    loss_history = k_data.get('loss_history', [])
    if loss_history:
        results['FEDS'] = {
            'rounds': list(range(1, len(loss_history) + 1)),
            'loss': [np.mean(l) for l in loss_history]
        }
        print(f"[RESULTS] Loaded FEDS: {len(loss_history)} rounds")

print(f"\n[RESULTS] Methods loaded: {list(results.keys())}")

In [ ]:
# Plot 1: Accuracy & Loss Comparison
print("[PLOT] Generating comparison plots...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
ax1 = axes[0]
for method, data in results.items():
    if 'rounds' in data and 'accuracy' in data:
        ax1.plot(data['rounds'], data['accuracy'], label=method, linewidth=2)
ax1.set_xlabel('Communication Round', fontsize=12)
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.set_title(f'Accuracy Comparison - {DATASET}', fontsize=14)
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)

# Loss plot
ax2 = axes[1]
for method, data in results.items():
    if 'rounds' in data and 'loss' in data:
        ax2.plot(data['rounds'], data['loss'], label=method, linewidth=2)
ax2.set_xlabel('Communication Round', fontsize=12)
ax2.set_ylabel('Loss', fontsize=12)
ax2.set_title(f'Loss Comparison - {DATASET}', fontsize=14)
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'benchmark_comparison_{DATASET}.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"[PLOT] Saved: benchmark_comparison_{DATASET}.png")

In [ ]:
# Plot 2: FEDS K Trajectory
if os.path.exists('feds_k_tracker.json'):
    print("[PLOT] Generating K trajectory plots...")
    with open('feds_k_tracker.json') as f:
        k_data = json.load(f)
    
    k_history = k_data.get('k_history', [])
    
    if k_history:
        k_array = np.array(k_history)
        rounds = np.arange(1, len(k_history) + 1)
        
        fig, axes = plt.subplots(2, 1, figsize=(12, 10))
        
        # Per-client K values
        ax1 = axes[0]
        for i in range(min(10, k_array.shape[1])):
            ax1.plot(rounds, k_array[:, i], alpha=0.6, linewidth=1.5, label=f'Client {i}')
        ax1.set_xlabel('Round', fontsize=12)
        ax1.set_ylabel('K Value', fontsize=12)
        ax1.set_title('FEDS: Per-Client Adaptive K Values', fontsize=14)
        ax1.legend(loc='upper right', fontsize=9, ncol=2)
        ax1.grid(True, alpha=0.3)
        
        # K statistics
        ax2 = axes[1]
        k_mean = k_array.mean(axis=1)
        k_std = k_array.std(axis=1)
        ax2.plot(rounds, k_mean, 'b-', linewidth=2.5, label='Mean K')
        ax2.fill_between(rounds, k_mean - k_std, k_mean + k_std, alpha=0.2, color='blue', label='±1 Std')
        ax2.plot(rounds, k_array.min(axis=1), 'r--', alpha=0.7, label='Min K')
        ax2.plot(rounds, k_array.max(axis=1), 'g--', alpha=0.7, label='Max K')
        ax2.set_xlabel('Round', fontsize=12)
        ax2.set_ylabel('K Value', fontsize=12)
        ax2.set_title('FEDS: K Statistics Across Clients', fontsize=14)
        ax2.legend(loc='upper right')
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f'feds_k_trajectory_{DATASET}.png', dpi=150, bbox_inches='tight')
        plt.show()
        print(f"[PLOT] Saved: feds_k_trajectory_{DATASET}.png")
else:
    print("[PLOT] No FEDS K trajectory data available")

In [ ]:
# Final Summary Table
print("\n" + "="*70)
print(f"  FINAL RESULTS SUMMARY - {DATASET}")
print("="*70)

print(f"\n{'Method':<15} {'Final Accuracy':<20} {'Final Loss':<20} {'Communication'}")
print("-"*70)

for method, data in results.items():
    if 'accuracy' in data and 'loss' in data:
        final_acc = data['accuracy'][-1] if data['accuracy'] else 0
        final_loss = data['loss'][-1] if data['loss'] else 0
        comm = "100%" if method == "FedAvg" else "~50% (adaptive)" if method == "FEDS" else "~50% (fixed)"
        print(f"{method:<15} {final_acc:<20.4f} {final_loss:<20.4f} {comm}")
    elif 'loss' in data:
        final_loss = data['loss'][-1] if data['loss'] else 0
        comm = "~50% (adaptive)"
        print(f"{method:<15} {'N/A':<20} {final_loss:<20.4f} {comm}")

if os.path.exists('feds_k_tracker.json'):
    with open('feds_k_tracker.json') as f:
        k_data = json.load(f)
    k_list = k_data.get('k_list', [])
    if k_list:
        total_params = max(k_list) * 2
        avg_k = sum(k_list) / len(k_list)
        savings = (1 - avg_k / total_params) * 100
        print(f"\nFEDS Average K: {avg_k:.0f}")
        print(f"FEDS Communication Savings: ~{savings:.1f}%")

print("\n" + "="*70)
print("  BENCHMARK COMPLETE!")
print("="*70)
print("\nGenerated files:")
print(f"  - benchmark_comparison_{DATASET}.png")
print(f"  - feds_k_trajectory_{DATASET}.png")